# Check the pancreas dataset

Open this notebook in Google Colab. Put `config.py`, `check_environment.py`, `prepare_data.py`, `requirements.txt`, and the extracted `Task07_Pancreas` folder inside `My Drive/Major`. Keep an existing `splits` folder there too; do not discard a saved split. Only change the project path in C1 if your Drive folder is different.

Run C1–C6 for setup, then C7–C9 for the real audit, saved patient split, and training examples. The audit code lives in `prepare_data.py`. This notebook runs in Colab, not on the project computer. A CPU Colab runtime is enough for Phase 3. No model is trained here.


## C1. Project location

Run cells from top to bottom in Colab. Put the project files in `My Drive/Major`, or change the Drive path in this cell. This sets the one project-root value used by `config.py`.

In [ ]:
import os
import sys
from pathlib import Path

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    raise RuntimeError("Open this notebook in Google Colab; local execution is disabled.")

# Change this Drive path if your project is in another Drive folder.
PROJECT_ROOT = Path("/content/drive/MyDrive/Major").expanduser()
PROJECT_ROOT = PROJECT_ROOT.resolve()
print("Project folder:", PROJECT_ROOT)
print("Running in Colab:", IN_COLAB)

## C2. Connect Drive and find the project

Colab will ask you to connect your Drive. Keep `config.py`, `check_environment.py`, `prepare_data.py`, and `requirements.txt` together in the Drive project folder. The dataset folder belongs beside them. Nothing is read from the project computer.

In [ ]:
if IN_COLAB:
    drive.mount("/content/drive")

required_files = ("config.py", "check_environment.py", "prepare_data.py", "dataset.py", "requirements.txt")
missing_files = [name for name in required_files if not (PROJECT_ROOT / name).is_file()]
if missing_files:
    raise FileNotFoundError(
        f"Missing {missing_files} in {PROJECT_ROOT}. "
        "Copy the project setup files there, or correct the path in C1."
    )

os.environ["PANCREAS_PROJECT_ROOT"] = str(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("Project setup files found.")

## C3. Install the project libraries

Install into this notebook's Python environment. This command keeps already installed packages when their requirements are satisfied. If installation asks for a runtime restart, restart it and run C1–C5 again.

In [ ]:
import subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
    "-r", str(PROJECT_ROOT / "requirements.txt"),
])

## C4. Load the shared paths

The Python files and both notebooks use `config.py`. Only the folders needed for setup and data preparation are created now. Later model and training code will live in Python files imported by the notebooks.

In [ ]:
import importlib
import config

importlib.reload(config)
config.prepare_folders()
print("Source dataset:", config.DATASET_DIR)
print("Prepared data:", config.PROCESSED_DIR)
print("Split lists:", config.SPLITS_DIR)
print("Persistent results:", config.RESULTS_DIR)
print("Working dataset:", config.RUNTIME_DATASET_DIR)

## C5. Check imports and hardware

This runs the shared checker, including a small PyTorch operation. It saves actual versions, RAM, disk space, and CUDA status in `results/environment_local.json` or `results/environment_colab.json`. CPU setup can pass without a GPU; GPU execution is a separate check.

In [ ]:
import check_environment

importlib.reload(check_environment)
environment = check_environment.run_checks()
if not environment["checks_passed"]:
    raise RuntimeError(
        "Environment checks failed. Review the printed errors and saved JSON report."
    )

## C6. Check the expected dataset location

This checks file locations only. The real annotation, alignment, and patient-split audit belongs to Phase 3. Missing data does not prevent the setup checks above.

In [ ]:
expected_parts = (
    config.DATASET_DIR / "dataset.json",
    config.DATASET_DIR / "imagesTr",
    config.DATASET_DIR / "labelsTr",
    config.DATASET_DIR / "imagesTs",
)
for path in expected_parts:
    print(f"{path.name}: {'present' if path.exists() else 'missing'}")
dataset_present = all(path.exists() for path in expected_parts)
print("Expected dataset locations present:", dataset_present)
print("Continue with C7 for the content audit and saved split.")

## C7. Audit the actual volumes and save patient lists

The audit checks the file list before opening each CT. Reading the volumes can take several minutes over Drive. If the runtime disconnects, rerun C1–C7. C7 checks the saved file hashes and reuses unchanged scans from the last completed group of ten. It then checks duplicates across all cases before saving the split.

Only usable labelled cases enter our seed-42 split. One MSD case ID is treated as one patient; no independent patient linkage table is supplied. A different frozen split is never overwritten automatically. Unlabelled `imagesTs` files do not enter the scored split.


In [ ]:
import prepare_data

importlib.reload(prepare_data)
if not dataset_present:
    raise FileNotFoundError(f"Dataset parts are missing from {config.DATASET_DIR}")
audit = prepare_data.run_audit()


## C8. Read and check the saved lists

Paths are relative to the dataset folder. The 70/15/15 split is our project protocol, giving 197/42/42 when all 281 cases are usable. Tumour presence and broad volume groups are balanced. Reference masks are used for stratification, not for inference.


In [ ]:
from dataset import read_splits

saved_splits = read_splits()
for name, cases in saved_splits.items():
    print(name, len(cases), "cases")
print("The saved case counts and hashes match the completed audit.")


## C9. Inspect training masks

These are reference-mask quality checks, not model predictions. Green marks pancreas and red marks tumour/mass. Examples include the smallest positive training mass and a peripheral slice where it is small. The CT display window is only for these pictures; training preprocessing is not selected here. No held-out masks are shown.


In [ ]:
from IPython.display import Image, display

import json
audit = json.loads((config.RESULTS_DIR / "data_audit.json").read_text(encoding="utf-8"))
for filename in audit["overlay_files"]:
    display(Image(filename=str(config.RESULTS_DIR / "data_overlays" / filename)))


## Continue with Phase 4

Read `results/data_audit.json` and any exclusion/warning rows before preparing the data. Keep the saved patient lists fixed. Phase 4 adds aligned preparation, a simple loader, and original-grid restoration checks.
